# 수학모델링캠프 — 시작하기

지렁이 거대신경섬유 축삭을 시뮬레이션하고, 중금속 세 종의 효과를 파형에서 읽는다.

**이 노트북은 얇은 운전대다.** 계산은 전부 `코드/src/` 의 `.py` 안에 있고,
여기서는 부르고 그리기만 한다. 이유는 `정리/수치해법과_코랩.md` 참조.

설명 문서: `수학모델링캠프/README.md`


## 1. 준비 — 저장소를 받고 경로와 한글 폰트를 잡는다


In [ ]:
# 아직 기본 브랜치에 병합되지 않았으므로 브랜치를 지정한다.
# 병합된 뒤에는 -b 이하를 빼면 된다.
!git clone -q -b claude/heavy-metal-neuron-sim-yobchu https://github.com/ianshin123/STSY.git

import sys
sys.path.insert(0, '/content/STSY/수학모델링캠프/코드/src')
import colab
ROOT = colab.setup()      # 경로 + 나눔고딕
print('준비 완료:', ROOT)


## 2. 스파이크를 하나 달리게 한다

`dt` 를 안 주면 안정 한계에 맞춰 알아서 잡는다.
한계를 넘기면 nan 대신 설명이 있는 오류가 난다.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hh, axon as A, recording as R, metals as M

ax = A.Axon(diameter_um=70.0, length_mm=60.0, dx_um=100.0)
print(f'구획 {ax.n_comp}개 · dt 안정 한계 {ax.max_stable_dt()*1000:.2f} µs')

run = A.simulate(ax, temp_c=20.0, t_end_ms=20.0,
                 stim_ua_cm2=300.0, stim_dur_ms=0.5, stim_comps=20,
                 record_every=2)

print('끝까지 전파:', run.propagated())
print('전도속도: %.3f m/s' % run.conduction_velocity(2.125, 4.625))


### 막전위가 축삭을 따라 달리는 모습


In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
for x_cm in (1.0, 2.0, 3.0, 4.0, 5.0):
    i = int(np.argmin(np.abs(run.x_cm - x_cm)))
    ax1.plot(run.t_ms, run.v[:, i], label=f'x = {x_cm:.0f} cm')
ax1.set_xlabel('시간 (ms)'); ax1.set_ylabel('막전위 (mV)')
ax1.set_title('활동전위가 지나가는 시각이 위치마다 밀린다')
ax1.legend(); ax1.grid(alpha=0.3); plt.show()


## 3. 전극이 보는 파형 — 여기서부터가 실험과 같은 것이다

막전위는 우리가 못 본다. 우리가 보는 것은 몸 표면 두 접점 사이의 전위차다.


In [ ]:
t, ch_a, ch_b, d_mm = R.channel_pair(run, span_mm=25.0, x_start_cm=1.5)

fig, ax2 = plt.subplots(figsize=(9, 4))
ax2.plot(t, ch_a, label='채널 A (앞)')
ax2.plot(t, ch_b, label='채널 B (뒤)')
ax2.set_xlabel('시간 (ms)'); ax2.set_ylabel('전위 (µV)')
ax2.set_title(f'이극 기록 · 접점 중심 간격 {d_mm:.1f} mm')
ax2.legend(); ax2.grid(alpha=0.3); plt.show()

print('진폭 peak-to-peak: %.1f µV   (문헌 MGF 29.4 µV · Yoshida 2009)' % np.ptp(ch_a))


## 4. 저장소의 분석 코드에 그대로 넣어 본다

`분석/src/velocity.py` 는 실제 측정 데이터를 처리하려고 만든 코드다.
시뮬레이션 파형을 192 kHz 로 다시 찍어 넣으면 참값을 되찾아야 한다.


In [ ]:
import velocity as V

FS = 192_000.0
_, a = R.resample(t, ch_a, FS)
_, b = R.resample(t, ch_b, FS)
a = R.add_noise(a, 19.3, seed=1)     # Yoshida 실측 잡음
b = R.add_noise(b, 19.3, seed=2)

est = V.estimate_dt(V.bandpass(a, FS), V.bandpass(b, FS), FS)
v_est = V.velocity(d_mm, est.dt_s)
v_true = run.conduction_velocity(2.125, 4.625)
print(f'참값 {v_true:.3f} m/s · 추정 {v_est:.3f} m/s · 오차 {100*(v_est-v_true)/v_true:+.3f} %')


## 5. 중금속 — 세 금속의 지문

납·카드뮴은 통로를 막고(차단형), 철은 막을 손상시킨다(손상형).
**구조가 다르므로 파형에 남는 자국도 다르다.**

근거와 한계는 `정리/금속별_기전.md`. **상수는 전부 우리 가정이다.**


In [ ]:
def measure(membrane=None, temp_c=20.0):
    a = A.Axon(diameter_um=70.0, length_mm=60.0, dx_um=100.0,
               membrane=membrane or hh.Membrane())
    r = A.simulate(a, temp_c, t_end_ms=30.0, stim_ua_cm2=300.0,
                   stim_dur_ms=0.5, stim_comps=20, record_every=4)
    return r.conduction_velocity(2.125, 4.625), float(np.ptp(R.bipolar(r, 2.0, 3.25)))

base = hh.Membrane()
v0, amp0 = measure()

print('요인          속도변화%  진폭변화%   S')
rows = [(m.name, m.membrane(base, 2.0, days=7.0), 20.0) for m in M.ALL]
rows += [('온도 -2도', base, 18.0), ('온도 +2도', base, 22.0)]
for name, mem, T in rows:
    v, amp = measure(mem, T)
    dv, da = 100*(v-v0)/v0, 100*(amp-amp0)/amp0
    print(f'{name:10s}  {dv:+8.2f}  {da:+8.2f}  {da/dv:+7.3f}')


**S = 진폭변화 / 속도변화** 하나로 네 요인이 갈린다.

```
  납        카드뮴      온도         철
 ─────────┼─────────┼───────────┼──────────
  S < 0    S ≈ 0     0.25~0.65    S > 1
```

한계는 `정리/파형에서_지표까지.md` 「한계」 절을 반드시 읽을 것.


## 6. 문서의 표를 통째로 다시 뽑기

몇 분 걸린다. 표를 고칠 일이 생기면 문서가 아니라 `sweep.py` 를 고친다.


In [ ]:
# !python3 /content/STSY/수학모델링캠프/코드/src/sweep.py
